# Model Verification Suite

This notebook provides a visual and numerical verification of the SEIXHRD model. It's designed to be run as a sanity check alongside the test suite to ensure things like population conservation, compartment flows, and vaccine logic are working as expected.

**Checks Performed:**
1.  **Mass Conservation**: Does the total population behave as expected (constant or strictly following birth/death rates)?
2.  **Compartment Flows**: Do individuals move correctly from S -> E -> I -> ...?
3.  **Vaccine Logic**: Does 100% perfect vaccination actually stop transmission?
4.  **Capacity Gating**: Does the Hill function correctly throttle admissions when capacity is exceeded?

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from simulate_model import simulate_model
from scenario_helpers import get_scenario_params
from scenarios import SCENARIO_REGISTRY, AGE_PARAMS_DEFAULT, AGE_POPS_DEFAULT, CONTACT_MATRIX_DEFAULT

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print("Imports successful.")

ModuleNotFoundError: No module named 'simulate_model'

## 1. Mass Conservation Check

We run the 'Baseline' scenario and sum all compartments at every time step. 
Since the baseline scenario has no births or background deaths (usually), the total population should be constant (within floating point error).

In [ ]:
params = get_scenario_params('baseline')
results = simulate_model(**params)

t = results['times']
total_pop = np.array(results['live_population']) + np.array(results['D_total']) + np.array(results['D_vax_total'])
initial_pop = total_pop[0]

plt.figure()
plt.plot(t, total_pop, label='Total Population')
plt.ylim(initial_pop * 0.99, initial_pop * 1.01)
plt.title(f"Mass Conservation Check (Initial Pop: {initial_pop:.0f})")
plt.xlabel("Time (days)")
plt.ylabel("Population")
plt.legend()
plt.show()

max_deviation = np.max(np.abs(total_pop - initial_pop))
print(f"Max deviation from initial population: {max_deviation:.2e}")
assert max_deviation < 1e-5, "Mass conservation failed!"

## 2. Compartment Flow Visualization

Visualizing the standard SEIR curves to ensure shapes look like other models.

In [ ]:
plt.figure()
plt.plot(t, np.sum(results['S'], axis=0), label='Susceptible')
plt.plot(t, results['E_total'], label='Exposed')
plt.plot(t, results['I_total'], label='Infected')
plt.plot(t, np.sum(results['R'], axis=0), label='Recovered')
plt.title("Aggregate SEIR Dynamics")
plt.xlabel("Time (days)")
plt.ylabel("Population")
plt.legend()
plt.show()

## 3. Vaccine Efficacy Logic

We compare a 'No Vaccine' scenario vs an 'Perfect Vaccine' scenario (100% coverage, 100% efficacy).
The Ideal Vaccine scenario should show ZERO infections if started before the outbreak.

In [ ]:
# No Vaccine
params_no_vax = get_scenario_params('baseline')
params_no_vax['vaccine_config']['coverage'] = [0.0, 0.0, 0.0]
res_no_vax = simulate_model(**params_no_vax)

# Ideal Vaccine (100% coverage, perfect immunity)
params_vax = get_scenario_params('baseline')
params_vax['vaccine_config']['coverage'] = [1.0, 1.0, 1.0]
params_vax['vaccine_config']['VE_infection'] = 1.0
res_vax = simulate_model(**params_vax)

total_infected_no_vax = np.sum(res_no_vax['I_total'])
total_infected_vax = np.sum(res_vax['I_total']) # Should be close to initial I0

print(f"Total Infected-Days (No Vax): {total_infected_no_vax:.0f}")
print(f"Total Infected-Days (Perfect Vax): {total_infected_vax:.0f}")

plt.figure()
plt.plot(res_no_vax['times'], res_no_vax['I_total'], label='No Vaccine')
plt.plot(res_vax['times'], res_vax['I_total'], label='Perfect Vaccine')
plt.title("Vaccine Impact Verification")
plt.xlabel("Time (days)")
plt.ylabel("Active Infections")
plt.legend()
plt.show()